In [1]:
import os
from tika import parser
from pathlib import Path
from tqdm.notebook import tqdm

from langchain_community.document_loaders import (
    PDFPlumberLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader,
)

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TesseractCliOcrOptions,
)
from langchain_docling.loader import DoclingLoader, ExportType
from docling.document_converter import DocumentConverter, PdfFormatOption

from polylex_chatbot.env import load_project_env

/home/saskya/dev/tb/polylex-chatbot/.venv/lib/python3.12/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [2]:
root_path = Path.cwd().parent.parent
env_file = load_project_env()
corpus_name = os.getenv("CORPUS_NAME")
documents_path = Path(os.path.join(root_path, "documents", corpus_name))

results_path = Path("comparison_results")
results_path.mkdir(parents=True, exist_ok=True)

loaders = [PDFPlumberLoader, PyMuPDFLoader, UnstructuredPDFLoader, DoclingLoader]

files = [
    {
        "filename": "53d04f32e0163590885f609000064f41.pdf",
        "reason": "pdf correspondant au template Fedlex avec references en bas de page (derniere lex mise a jour en date du 7 avril 2026)"
    },
    {
        "filename": "837ffe5759b8898b2cbbdd1e577773b8.pdf",
        "reason": "pdf correspondant au template Polylex"
    },
    {
        "filename": "5f909c5f962b5e6f610527d2469d8dfb.pdf",
        "reason": "pdf avec graphiques, diagrammes et structure complexe (derniere lex ajoutee en date du 7 avril 2026)"
    },
    {
        "filename": "78199ce04a046cec1bbad1fc9bac50d6.pdf",
        "reason": "pdf avec du texte completement scanne"
    },
    {
        "filename": "3bbe7877e0bdafebf848b6a0ba0aacab.pdf",
        "reason": "pdf avec un tableau basique et du texte en anglais"
    },
    {
        "filename": "3976ecec7bf1095443bc5fb7a9493f38.pdf",
        "reason": "pdf avec differents types de tableaux + table des matieres"
    },
    {
        "filename": "c1feac5448601b891e68757083811224.pdf",
        "reason": "pdf contenant egalement de l'allemand"
    }
]

loaders = [
    (PDFPlumberLoader,
     {
         "extract_images": False, # error inside PDFPlumber otherwise
         "dedupe": True,
         "text_kwargs": {
             "layout": False,
             "x_tolerance": 2,
             "y_tolerance": 2,
             "use_text_flow": True
         }
     }
    ),
    (PyMuPDFLoader, {"mode": "page", "extract_images": True, "extract_tables": "csv"}), # better page instead of single
    (UnstructuredPDFLoader, {"mode": "elements", "languages": ["fra", "eng"]}), # better elements instead of single
]

In [3]:
pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=TesseractCliOcrOptions(
        lang=["fra", "eng"],
        force_full_page_ocr=True
    ),
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

docling = (DoclingLoader, {"converter": converter, "export_type": ExportType.DOC_CHUNKS}) # too small chunks but can be merged (custom)
loaders.append(docling)

In [4]:
for loader_cls, kwargs in tqdm(loaders):
    loader_output_dir = results_path / f"{loader_cls.__name__}_results"
    loader_output_dir.mkdir(parents=True, exist_ok=True)
    for file in tqdm(files):
        path = documents_path / Path(file["filename"])
        doc_id = path.stem
        loader = loader_cls(str(path), **kwargs)
        docs = loader.load()
        extracted_text = "\n\n".join(doc.page_content for doc in docs)
        output_file = loader_output_dir / f"{doc_id}.txt"
        output_file.write_text(extracted_text, encoding="utf-8")

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
OSD failed (doc 53d04f32e0163590885f609000064f41.pdf, page: 2, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x78916d3b0140>):
 b'Estimating resolution as 233\nToo few characters. Skipping this page\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'
Token indices sequence length is longer than the specified maximum sequence length for this model (2620 > 512). Running this sequence through the model will result in indexing errors
OSD failed (doc 3976ecec7bf1095443bc5fb7a9493f38.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x7891722efb30>):
 b'Estimating resolution as 697\nWarning. Invalid resolution 0 dpi. Using 70 instead.\nToo few characters. Skipping this page\nError during processing.\n'
Token indices sequence len

In [5]:
for file in tqdm(files):
    output_dir = results_path / "Tika_results"
    output_dir.mkdir(parents=True, exist_ok=True)
    path = documents_path / file["filename"]
    doc_id = path.stem
    parsed_pdf = parser.from_file(str(path))
    extracted_text = parsed_pdf['content']
    output_file = output_dir / f"{doc_id}.txt"
    output_file.write_text(extracted_text, encoding="utf-8")

  0%|          | 0/7 [00:00<?, ?it/s]